# Xori Local GPT — обучение в Google Colab\n\nНебольшой генеративный Transformer обучается с нуля на датасете Xori. Colab даёт GPU для обучения, затем модель экспортируется в ONNX и запускается в Node.js через ONNX Runtime.\n\nПервая версия небольшая: качество будет расти вместе с качеством и объёмом датасета.

In [ ]:
!pip -q install onnx onnxruntime\nimport torch, json, random\nfrom pathlib import Path\nprint('PyTorch:',torch.__version__)\nprint('CUDA:',torch.cuda.is_available())\nif torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
DATA_URL='https://raw.githubusercontent.com/xoristalin-dotcom/Xori-/main/xori_dataset.jsonl'\n!wget -q -O xori_dataset.jsonl $DATA_URL\nrows=[json.loads(x) for x in open('xori_dataset.jsonl',encoding='utf-8') if x.strip()]\ntexts=[r['text'] for r in rows]\nprint('examples:',len(texts))

In [ ]:
SPECIAL=['<pad>','<bos>','<eos>','<unk>']\nchars=sorted(set(''.join(texts)))\nitos=SPECIAL+chars\nstoi={ch:i for i,ch in enumerate(itos)}\nPAD,BOS,EOS,UNK=[stoi[x] for x in SPECIAL]\nMAX_LEN=256\ndef encode(s): return [BOS]+[stoi.get(c,UNK) for c in s]+[EOS]\nprint('vocab:',len(itos))

In [ ]:
class XoriGPT(torch.nn.Module):\n    def __init__(self,vocab_size,d_model=192,n_heads=6,n_layers=4,max_len=256):\n        super().__init__()\n        self.tok=torch.nn.Embedding(vocab_size,d_model)\n        self.pos=torch.nn.Embedding(max_len,d_model)\n        layer=torch.nn.TransformerEncoderLayer(d_model=d_model,nhead=n_heads,dim_feedforward=d_model*4,dropout=0.0,activation='gelu',batch_first=True,norm_first=True)\n        self.blocks=torch.nn.TransformerEncoder(layer,num_layers=n_layers)\n        self.norm=torch.nn.LayerNorm(d_model)\n        self.lm_head=torch.nn.Linear(d_model,vocab_size,bias=False)\n        mask=torch.triu(torch.ones(max_len,max_len,dtype=torch.bool),diagonal=1)\n        self.register_buffer('mask',mask,persistent=False)\n    def forward(self,input_ids):\n        B,T=input_ids.shape\n        pos=torch.arange(T,device=input_ids.device).unsqueeze(0)\n        x=self.tok(input_ids)+self.pos(pos)\n        x=self.blocks(x,mask=self.mask[:T,:T])\n        return self.lm_head(self.norm(x))\n

In [ ]:
def make_batch(batch_rows):\n    seqs=[encode(s)[:MAX_LEN] for s in batch_rows]\n    x=torch.full((len(seqs),MAX_LEN),PAD,dtype=torch.long)\n    for i,s in enumerate(seqs): x[i,:len(s)]=torch.tensor(s)\n    return x\ndevice='cuda' if torch.cuda.is_available() else 'cpu'\nmodel=XoriGPT(len(itos)).to(device)\nopt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=0.01)\nloss_fn=torch.nn.CrossEntropyLoss(ignore_index=PAD)\nprint('parameters:',sum(p.numel() for p in model.parameters()))\nprint('device:',device)

In [ ]:
EPOCHS=35\nBATCH=16\nfor epoch in range(EPOCHS):\n    random.shuffle(texts)\n    model.train(); total=0.0; count=0\n    for start in range(0,len(texts),BATCH):\n        x=make_batch(texts[start:start+BATCH]).to(device)\n        logits=model(x)\n        loss=loss_fn(logits[:,:-1].reshape(-1,len(itos)),x[:,1:].reshape(-1))\n        opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()\n        total+=loss.item(); count+=1\n    if (epoch+1)%5==0 or epoch==0: print(f'epoch {epoch+1}/{EPOCHS} loss={total/max(1,count):.4f}')

In [ ]:
OUT=Path('xori_model'); OUT.mkdir(exist_ok=True)\nmodel.eval().cpu()\ndummy=torch.full((1,8),PAD,dtype=torch.long)\ntorch.onnx.export(model,dummy,OUT/'model.onnx',input_names=['input_ids'],output_names=['logits'],dynamic_axes={'input_ids':{1:'sequence'},'logits':{1:'sequence'}},opset_version=17,dynamo=False)\njson.dump({ch:i for i,ch in enumerate(itos)},open(OUT/'vocab.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)\njson.dump({'version':1,'maxSeqLen':MAX_LEN,'bosId':BOS,'eosId':EOS,'unkId':UNK,'vocabSize':len(itos)},open(OUT/'config.json','w',encoding='utf-8'),indent=2)\nprint('exported:',list(OUT.iterdir()))

In [ ]:
import onnxruntime as ort\nsess=ort.InferenceSession(str(OUT/'model.onnx'),providers=['CPUExecutionProvider'])\nprint(sess.get_inputs()[0].name,sess.get_outputs()[0].name)\nprint('ONNX export OK')

## Перенос в GitHub\nСкачай папку `xori_model` из Colab и добавь её в корень репозитория. Бинарные артефакты сейчас исключены из Git, чтобы не раздувать репозиторий. Для Render модель нужно разместить отдельно или убрать эти три строки из `.gitignore`.

In [ ]:
!zip -qr xori_model.zip xori_model\nfrom google.colab import files\nfiles.download('xori_model.zip')